### 1. Inference Model: phoBert

More demo in google colab

<a href="https://colab.research.google.com/drive/1LGYA5dgTkTZfnY2D_tkqR9uT9MV47PNC#scrollTo=yI2e1sIf-ngx" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#### Inference - Sử dụng phoBert - Sentiments classification

- Inference task gồm 3 bước chính: tiền xử lý dữ liệu đầu vào (preprocessing), chạy mô hình dự đoán (inference), và hậu xử lý kết quả (postprocessing).

- Vì bài toán phân loại cảm xúc : tích cực, trung tính và tiêu cực => lựa chọn text-classification

#### 1. Data Preparation

In [1]:
# sử dụng data để inference và evaluation
data = "Tôi luôn tự hào với bạn bè về gia đình của mình. Nhà tôi giàu có lắm nhưng không phải giàu vì tiền bạc mà giàu bởi tình cảm. Có được điều đáng quý này phải kể đến sự góp công rất lớn của cha. Gia đình muốn giữ được lửa hạnh phúc cần có sự đồng cảm, quan tâm và sẻ chia. Cả ngày đi làm vất vả nhưng cha vẫn luôn sẵn sàng đỡ đần mẹ công việc nhà. Tôi biết công việc của cha cũng nhiều khó khăn và áp lực, nhưng đến khi bước chân vào cánh cổng mọi buồn bực ưu phiền cha đều gác lại, để cho nụ cười hé nở trên môi. Có nhiều lần tôi bắt gặp nét âu sầu trên khuôn mặt cha, thế nhưng khi tôi hỏi cha lại tỏ ra vui vẻ như không có chuyện gì. Cha tôi là vậy đấy, luôn đem niềm vui, sự ân cần tin tưởng cho người khác còn nỗi buồn chỉ để cho riêng mình. Tôi thương và trân quý tấm lòng của cha vô cùng."

#### 2. Load model

In [2]:
#!pip install transformers torch pyvi

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from pyvi import ViTokenizer

model_name = "wonrax/phobert-base-vietnamese-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)



/Users/theson/miniforge3/envs/AIO/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# xử lý với Vitokenizer
segmented = ViTokenizer.tokenize(data)
print(f"Segmented: {segmented}")

# Encode
input_ids = tokenizer.encode(segmented, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

print("Danh sách token (subword):")
for i, tok in enumerate(tokens):
    print(f"{i:02d}: {tok}")

Segmented: Tôi luôn tự_hào với bạn_bè về gia_đình của mình . Nhà tôi giàu_có lắm nhưng không phải giàu vì tiền_bạc mà giàu bởi tình_cảm . Có được điều đáng quý này phải kể đến sự góp công rất lớn của cha . Gia_đình muốn giữ được lửa hạnh_phúc cần có sự đồng_cảm , quan_tâm và sẻ chia . Cả ngày đi làm vất_vả nhưng cha vẫn luôn sẵn_sàng đỡ_đần mẹ công_việc nhà . Tôi biết công_việc của cha cũng nhiều khó_khăn và áp_lực , nhưng đến khi bước chân vào cánh cổng mọi buồn_bực ưu phiền cha đều gác lại , để cho nụ cười hé nở trên môi . Có nhiều lần tôi bắt_gặp nét âu_sầu trên khuôn_mặt cha , thế nhưng khi tôi hỏi cha lại tỏ ra vui_vẻ như_không có chuyện gì . Cha tôi là vậy đấy , luôn đem niềm vui , sự ân_cần tin_tưởng cho người khác còn nỗi buồn chỉ để cho riêng mình . Tôi thương và trân quý tấm lòng của cha vô_cùng .
Danh sách token (subword):
00: <s>
01: Tôi
02: luôn
03: tự_hào
04: với
05: bạn_bè
06: về
07: gia_đình
08: của
09: mình
10: .
11: Nhà
12: tôi
13: giàu_có
14: lắm
15: nhưng
16: không


In [4]:
# Inference
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)[0]

print("Kết quả phân loại cảm xúc:")
print("="*30)

# Hiển thị xác suất cho tất cả các class
for idx in range(len(probs)):
    label = model.config.id2label[idx]
    score = probs[idx].item()
    bar = "█" * int(score * 50)
    print(f"{label:10s}: {score:.4f} |{bar}")

# Prediction cuối cùng
predicted_class = torch.argmax(probs).item()
predicted_label = model.config.id2label[predicted_class]
predicted_score = probs[predicted_class].item()

print("="*30)
print(f"Dự đoán: {predicted_label}")
print(f"Độ tin cậy: {predicted_score:.4f} ({predicted_score*100:.2f}%)")
print("="*30)



Kết quả phân loại cảm xúc:
NEG       : 0.0035 |
POS       : 0.9744 |████████████████████████████████████████████████
NEU       : 0.0220 |█
Dự đoán: POS
Độ tin cậy: 0.9744 (97.44%)
